In [9]:
import stim

In [10]:
import torch
import torch.nn.functional as F


In [ ]:
from torch_geometric.nn import MessagePassing, global_mean_pool
from torch_geometric.data import Data, Batch
from torch_geometric.utils import add_self_loops, degree


In [12]:
import matplotlib.pyplot as plt


In [25]:
import numpy as np
from tqdm.auto import trange
import networkx as nx
import math

In [14]:
# Set device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cuda


In [15]:
# ============================================================================
# PART 1: SURFACE CODE CIRCUIT
# ============================================================================

def surface_code_circuit(p, d):
    """Generate surface code circuit with given error rate and distance"""
    return stim.Circuit.generated(
        "surface_code:rotated_memory_z",
        rounds=d,
        distance=d,
        after_clifford_depolarization=p,
        after_reset_flip_probability=p,
        before_measure_flip_probability=p,
        before_round_data_depolarization=p
    )

In [ ]:
# ============================================================================
# PART 2: SYNDROME GRAPH - Pre-built structure, only values change
# ============================================================================
class SyndromeGraph:
    """
    Pre-built graph structure for a surface code.
    The skeleton (edges) is fixed. Only node values change per sample.
    """

    def __init__(self, circuit, device):
        self.device = device
        self.num_nodes = circuit.num_detectors

        # Build the fixed graph skeleton ONCE
        self.edge_index = self._build_edges(circuit).to(device)
        self.num_edges = self.edge_index.shape[1]

        print(f"✓ Graph skeleton built: {self.num_nodes} detectors, {self.num_edges} connections")
        print(f"  Average neighbors per detector: {self.num_edges / self.num_nodes:.1f}")

    def _build_edges(self, circuit):
        """
        Build edge connections based on detector positions.

        Edge types:
        1. Spatial (bidirectional): Connect detectors at same time that are
           adjacent in space (differ by 2 in exactly one of x or y)
        2. Temporal (unidirectional): Connect each detector to the same
           spatial position in the next time round (t -> t+1)
        """
        detector_coords = circuit.get_detector_coordinates()
        edges = []

        # Build lookup: (x, y, t) -> detector_id for temporal connections
        coord_to_id = {}
        for det_id, coord in detector_coords.items():
            key = (coord[0], coord[1], coord[2])
            coord_to_id[key] = det_id

        for i in range(self.num_nodes):
            coord_i = detector_coords.get(i, [0, 0, 0])
            xi, yi, ti = coord_i[0], coord_i[1], coord_i[2]

            for j in range(i + 1, self.num_nodes):
                coord_j = detector_coords.get(j, [0, 0, 0])
                xj, yj, tj = coord_j[0], coord_j[1], coord_j[2]

                # Spatial connection: same time round, adjacent in space
                # Adjacent means differ by 2 in exactly one coordinate
                if ti == tj:
                    dx = abs(xi - xj)
                    dy = abs(yi - yj)

                    # Check for up/down/left/right neighbors (not diagonal)
                    if (dx == 2 and dy == 0) or (dx == 0 and dy == 2):
                        edges.append([i, j])
                        edges.append([j, i])  # Bidirectional

            # Temporal connection: same spatial position, next time round
            # Unidirectional: only forward in time (t -> t+1)
            next_key = (xi, yi, ti + 1)
            if next_key in coord_to_id:
                j = coord_to_id[next_key]
                edges.append([i, j])  # Unidirectional forward in time

        if len(edges) == 0:
            print("  Warning: No edges found, using fallback chain")
            for i in range(self.num_nodes - 1):
                edges.append([i, i + 1])
                edges.append([i + 1, i])

        return torch.tensor(edges, dtype=torch.long).t().contiguous()

    def create_batch(self, syndrome_values):
        """
        Create batched graph data from syndrome measurements.
        EFFICIENT: Structure is pre-built, just plugs in new values.

        Args:
            syndrome_values: Tensor (batch_size, num_nodes) with +1/-1 values

        Returns:
            PyG Data object ready for GNN
        """
        batch_size = syndrome_values.shape[0]

        # Stack all node features: (batch_size * num_nodes, 1)
        x = syndrome_values[:, :self.num_nodes].reshape(-1, 1).float()

        # Replicate edges for each graph in batch (with offsets)
        edge_indices = []
        for i in range(batch_size):
            offset = i * self.num_nodes
            edge_indices.append(self.edge_index + offset)

        edge_index = torch.cat(edge_indices, dim=1)

        # Batch assignment: which graph each node belongs to
        batch = torch.arange(batch_size, device=self.device).repeat_interleave(self.num_nodes)

        return Data(x=x, edge_index=edge_index, batch=batch)

In [17]:
# ============================================================================
# PART 4: SIMPLE GCN LAYER (NO COMPILED DEPENDENCIES)
# ============================================================================

class SimpleGCNConv(MessagePassing):
    """Graph Convolutional Layer - works without pyg-lib"""

    def __init__(self, in_channels, out_channels):
        super().__init__(aggr='add')
        self.lin = torch.nn.Linear(in_channels, out_channels, bias=False)
        self.bias = torch.nn.Parameter(torch.Tensor(out_channels))
        self.reset_parameters()

    def reset_parameters(self):
        self.lin.reset_parameters()
        self.bias.data.zero_()

    def forward(self, x, edge_index):
        # Add self-loops to adjacency
        edge_index, _ = add_self_loops(edge_index, num_nodes=x.size(0))

        # Transform features
        x = self.lin(x)

        # Compute normalization (symmetric normalization)
        row, col = edge_index
        deg = degree(col, x.size(0), dtype=x.dtype)
        deg_inv_sqrt = deg.pow(-0.5)
        deg_inv_sqrt[deg_inv_sqrt == float('inf')] = 0
        norm = deg_inv_sqrt[row] * deg_inv_sqrt[col]

        # Message passing
        out = self.propagate(edge_index, x=x, norm=norm)

        return out + self.bias

    def message(self, x_j, norm):
        # Normalize messages from neighbors
        return norm.view(-1, 1) * x_j

In [ ]:
# ============================================================================
# PART 4: GNN DECODER MODEL
# ============================================================================

class SyndromeDecoder(torch.nn.Module):
    """
    GNN that takes syndrome graphs and predicts logical error.
    Each detector shares info with neighbors through message passing.
    """

    def __init__(self, hidden_dim=128, num_layers=4):
        super().__init__()

        # Message passing layers
        self.layers = torch.nn.ModuleList()
        self.norms = torch.nn.ModuleList()

        # First layer: 1 input (syndrome value) -> hidden
        self.layers.append(SimpleGCNConv(1, hidden_dim))
        self.norms.append(torch.nn.BatchNorm1d(hidden_dim))

        # Hidden layers
        for _ in range(num_layers - 1):
            self.layers.append(SimpleGCNConv(hidden_dim, hidden_dim))
            self.norms.append(torch.nn.BatchNorm1d(hidden_dim))

        # Output: aggregate graph -> predict error (logits, no sigmoid)
        self.output = torch.nn.Sequential(
            torch.nn.Linear(hidden_dim, 64),
            torch.nn.SiLU(),
            torch.nn.Linear(64, 1)
        )

    def forward(self, data):
        x, edge_index, batch = data.x, data.edge_index, data.batch

        # Message passing: nodes share info with neighbors
        for layer, norm in zip(self.layers, self.norms):
            x = layer(x, edge_index)
            x = norm(x)
            x = F.silu(x)

        # Pool: aggregate all nodes in each graph (efficient scatter-based)
        graph_features = global_mean_pool(x, batch)

        # Predict logical error
        return self.output(graph_features)

In [ ]:
# ============================================================================
# PART 5: GNN TRAINER - Clean interface with progress tracking
# ============================================================================

class GNNTrainer:
    """Clean interface for training GNN decoder with clear progress tracking"""

    def __init__(self, p, d, device, hidden_dim=128, num_layers=4):
        self.p = p
        self.d = d
        self.device = device

        print(f"\n{'='*60}")
        print(f"Initializing GNN Trainer for d={d}, p={p}")
        print(f"{'='*60}")

        # Build circuit (ONCE)
        print("\n[1/3] Building quantum circuit...")
        self.circuit = surface_code_circuit(p, d)

        # Build graph structure (ONCE)
        print("\n[2/3] Building graph skeleton...")
        self.graph = SyndromeGraph(self.circuit, device)

        # Build model
        print("\n[3/3] Building neural network...")
        self.model = SyndromeDecoder(hidden_dim=hidden_dim, num_layers=num_layers).to(device)
        self.optimizer = torch.optim.Adam(self.model.parameters(), lr=1e-3)
        # Note: loss_fn will be set dynamically in train_epoch to handle class imbalance

        # Count parameters
        num_params = sum(p.numel() for p in self.model.parameters())
        print(f"✓ Model built: {num_params:,} parameters")
        print(f"\n{'='*60}")
        print("Ready to train!")
        print(f"{'='*60}\n")

    def sample_data(self, num_samples):
        """Generate syndrome samples"""
        sampler = self.circuit.compile_detector_sampler()
        detections, flips = sampler.sample(num_samples, separate_observables=True)

        # Convert: 0/1 -> -1/+1
        # Use .copy() to ensure contiguous array, then torch.from_numpy() to avoid dtype inference issues
        syndrome_np = (detections.astype(np.int64) * 2 - 1).copy()
        labels_np = flips.astype(np.int64).flatten().copy()

        syndromes = torch.from_numpy(syndrome_np).to(self.device)
        labels = torch.from_numpy(labels_np).to(self.device)

        return syndromes, labels

    def train_epoch(self, syndromes, labels, batch_size=256):
        """One training epoch with progress bar"""
        self.model.train()
        num_samples = syndromes.shape[0]
        num_batches = math.ceil(num_samples / batch_size)

        # Compute class weights for imbalanced data
        # At low error rates, positive class (errors) is rare
        num_pos = labels.sum().item()
        num_neg = num_samples - num_pos
        if num_pos > 0 and num_neg > 0:
            pos_weight = torch.tensor([num_neg / num_pos], device=self.device)
        else:
            pos_weight = torch.tensor([1.0], device=self.device)
        loss_fn = torch.nn.BCEWithLogitsLoss(pos_weight=pos_weight)

        total_loss = 0
        correct = 0
        samples_seen = 0

        with trange(num_batches, desc="Training", leave=False) as pbar:
            for batch_idx in pbar:
                start = batch_idx * batch_size
                end = min(start + batch_size, num_samples)  # Handle last batch

                batch_syn = syndromes[start:end]
                batch_lab = labels[start:end]
                batch_len = end - start  # Actual batch size

                # Create graph batch (structure pre-built, just plug in values)
                batch_data = self.graph.create_batch(batch_syn)

                # Forward (get logits, not probabilities)
                logits = self.model(batch_data)
                loss = loss_fn(logits, batch_lab.unsqueeze(1).float())

                # Backward
                self.optimizer.zero_grad()
                loss.backward()
                self.optimizer.step()

                # Track metrics (convert logits to predictions)
                total_loss += loss.item()
                pred = torch.sigmoid(logits)
                batch_correct = ((pred > 0.5).flatten() == batch_lab).sum().item()
                correct += batch_correct
                samples_seen += batch_len

                # Update progress bar with correct running accuracy
                running_acc = correct / samples_seen
                pbar.set_postfix({
                    'loss': f'{loss.item():.4f}',
                    'acc': f'{running_acc:.4f}'
                })

        avg_loss = total_loss / num_batches
        accuracy = correct / num_samples  # Use actual sample count
        return avg_loss, accuracy

    def evaluate(self, num_samples=10000, batch_size=500):
        """Evaluate on fresh data"""
        self.model.eval()
        syndromes, labels = self.sample_data(num_samples)

        all_preds = []
        with torch.no_grad():
            for i in range(0, num_samples, batch_size):
                batch_syn = syndromes[i:i+batch_size]
                batch_data = self.graph.create_batch(batch_syn)
                logits = self.model(batch_data)
                pred = torch.sigmoid(logits)
                all_preds.append((pred > 0.5).flatten())

        all_preds = torch.cat(all_preds)
        accuracy = (all_preds == labels).float().mean().item()
        return accuracy

    def get_ler(self, num_shots=100000, batch_size=1000):
        """Compute logical error rate"""
        self.model.eval()
        syndromes, labels = self.sample_data(num_shots)

        all_preds = []
        with torch.no_grad():
            for i in range(0, num_shots, batch_size):
                batch_syn = syndromes[i:i+batch_size]
                batch_data = self.graph.create_batch(batch_syn)
                logits = self.model(batch_data)
                pred = torch.sigmoid(logits)
                all_preds.append((pred > 0.5).flatten().cpu().numpy())

        all_preds = np.concatenate(all_preds)
        errors = (all_preds != labels.cpu().numpy())
        return errors.mean()

    def save(self, path):
        """Save model weights"""
        torch.save(self.model.state_dict(), path)
        print(f"✓ Model saved to {path}")

    def load(self, path):
        """Load model weights"""
        self.model.load_state_dict(torch.load(path, weights_only=True))
        print(f"✓ Model loaded from {path}")

In [21]:
# ============================================================================
# PART 6: MWPM BASELINE FOR COMPARISON
# ============================================================================

def get_mwpm_accuracy(p, d, num_shots=100000):
    """Compute MWPM accuracy for comparison"""
    import pymatching

    print(f"Computing MWPM baseline ({num_shots:,} shots)...")

    circuit = surface_code_circuit(p, d)
    sampler = circuit.compile_detector_sampler()
    detection_events, observable_flips = sampler.sample(num_shots, separate_observables=True)

    detector_error_model = circuit.detector_error_model(decompose_errors=True)
    matcher = pymatching.Matching.from_detector_error_model(detector_error_model)

    predictions = matcher.decode_batch(detection_events)

    num_errors = sum(
        1 for shot in range(num_shots)
        if not np.array_equal(observable_flips[shot], predictions[shot])
    )

    ler = num_errors / num_shots
    accuracy = 1 - ler
    print(f"✓ MWPM Accuracy: {accuracy:.6f} (LER: {ler:.6f})")

    return accuracy

In [22]:
# ============================================================================
# PART 7: PROGRESSIVE TRAINING - Train until beating MWPM
# ============================================================================

def train_until_beat_mwpm(p, d, device, max_train_size=10**8, chunk_size=10**7):
    """Train with progressively larger datasets until beating MWPM"""
    import gc

    print(f"\n{'#'*60}")
    print(f"# PROGRESSIVE TRAINING: d={d}, p={p}")
    print(f"{'#'*60}")

    # Get MWPM baseline first
    print("\n" + "="*60)
    print("STEP 1: Getting MWPM baseline")
    print("="*60)
    mwpm_accuracy = get_mwpm_accuracy(p, d)
    print(f"\n🎯 Target to beat: {mwpm_accuracy:.6f}")

    # Initialize trainer
    print("\n" + "="*60)
    print("STEP 2: Setting up GNN")
    print("="*60)
    trainer = GNNTrainer(p, d, device)

    # Progressive training
    print("\n" + "="*60)
    print("STEP 3: Progressive training")
    print("="*60)

    train_size = 100
    beat_mwpm = False

    while train_size <= max_train_size and not beat_mwpm:
        print(f"\n{'─'*60}")
        print(f"📊 Training with {train_size:,} samples")
        print(f"{'─'*60}")

        # Reset model for fresh start
        trainer.model = SyndromeDecoder(hidden_dim=128, num_layers=4).to(device)
        trainer.optimizer = torch.optim.Adam(trainer.model.parameters(), lr=1e-3)

        # Calculate chunks needed
        num_chunks = max(1, train_size // chunk_size)
        samples_per_chunk = min(train_size, chunk_size)

        # Train in chunks
        for chunk_idx in range(num_chunks):
            current_chunk_size = min(samples_per_chunk, train_size - chunk_idx * chunk_size)

            print(f"\n  Chunk {chunk_idx + 1}/{num_chunks}: {current_chunk_size:,} samples")
            print(f"  Generating data...")

            syndromes, labels = trainer.sample_data(current_chunk_size)

            # Multiple epochs per chunk for small datasets
            num_epochs = max(1, min(10, 10000 // (current_chunk_size // 256 + 1)))

            for epoch in range(num_epochs):
                loss, acc = trainer.train_epoch(syndromes, labels, batch_size=256)
                print(f"    Epoch {epoch+1}/{num_epochs}: Loss={loss:.4f}, Train Acc={acc:.4f}")

            # Cleanup
            del syndromes, labels
            gc.collect()
            if torch.cuda.is_available():
                torch.cuda.empty_cache()

        # Evaluate
        print(f"\n  Evaluating on fresh test data...")
        gnn_accuracy = trainer.evaluate(num_samples=10000)

        print(f"\n  📈 Results:")
        print(f"     GNN Accuracy:  {gnn_accuracy:.6f}")
        print(f"     MWPM Accuracy: {mwpm_accuracy:.6f}")
        print(f"     Difference:    {(gnn_accuracy - mwpm_accuracy)*100:+.3f}%")

        if gnn_accuracy > mwpm_accuracy:
            print(f"\n  🎉 SUCCESS! GNN beats MWPM at train_size = {train_size:,}")
            beat_mwpm = True
        else:
            print(f"\n  ❌ Not yet. Trying 10x more data...")
            train_size *= 10
            gc.collect()
            if torch.cuda.is_available():
                torch.cuda.empty_cache()

    if not beat_mwpm:
        print(f"\n❌ Did not beat MWPM within max train_size = {max_train_size:,}")
        return None, None

    return trainer, train_size

In [ ]:
# ============================================================================
# PART 8: RUN THE EXPERIMENT
# ============================================================================

import os

# Configuration
train_p = 0.005
max_train_size = 10**8
chunk_size = 10**7

results = {}
trained_models = {}

# Train for each distance
for d in [3, 5, 7]:
    model_path = f"gnn_decoder_d{d}_p{train_p}.pt"

    print(f"\n{'█'*60}")
    print(f"█  DISTANCE {d}")
    print(f"{'█'*60}")

    # Check if model already exists
    if os.path.exists(model_path):
        print(f"\n✓ Found existing model: {model_path}")
        trainer = GNNTrainer(train_p, d, device)
        trainer.load(model_path)
        train_size = None
    else:
        print(f"\n→ No saved model found. Training new model...")
        trainer, train_size = train_until_beat_mwpm(
            p=train_p, d=d, device=device,
            max_train_size=max_train_size, chunk_size=chunk_size
        )

        if trainer is not None:
            trainer.save(model_path)
        else:
            print(f"Training failed for d={d}")
            continue

    trained_models[d] = trainer
    results[d] = train_size

# Print summary
print(f"\n\n{'═'*60}")
print("FINAL RESULTS SUMMARY")
print(f"{'═'*60}")
for d, train_size in results.items():
    if train_size:
        print(f"  Distance {d}: Beat MWPM with {train_size:,} samples")
    else:
        print(f"  Distance {d}: Loaded from saved model")
print(f"{'═'*60}")


████████████████████████████████████████████████████████████
█  DISTANCE 3
████████████████████████████████████████████████████████████

→ No saved model found. Training new model...

############################################################
# PROGRESSIVE TRAINING: d=3, p=0.005
############################################################

STEP 1: Getting MWPM baseline
Computing MWPM baseline (100,000 shots)...
✓ MWPM Accuracy: 0.982280 (LER: 0.017720)

🎯 Target to beat: 0.982280

STEP 2: Setting up GNN

Initializing GNN Trainer for d=3, p=0.005

[1/3] Building quantum circuit...

[2/3] Building graph skeleton...
✓ Graph skeleton built: 24 detectors, 48 connections
  Average neighbors per detector: 2.0

[3/3] Building neural network...
✓ Model built: 59,137 parameters

Ready to train!


STEP 3: Progressive training

────────────────────────────────────────────────────────────
📊 Training with 100 samples
────────────────────────────────────────────────────────────

  Chunk 1/1: 100 s

    Epoch 1/10: Loss=0.7472, Train Acc=0.0700


    Epoch 2/10: Loss=0.7162, Train Acc=0.1300


    Epoch 3/10: Loss=0.6885, Train Acc=0.2600


    Epoch 4/10: Loss=0.6621, Train Acc=0.8400


...

    Epoch 8/10: Loss=0.2772, Train Acc=0.8967


    Epoch 9/10: Loss=0.2763, Train Acc=0.8967


    Epoch 10/10: Loss=0.2748, Train Acc=0.8967

  Evaluating on fresh test data...

  📈 Results:
     GNN Accuracy:  0.896700
     MWPM Accuracy: 0.982280
     Difference:    -8.558%

  ❌ Not yet. Trying 10x more data...

────────────────────────────────────────────────────────────
📊 Training with 100,000 samples
────────────────────────────────────────────────────────────

  Chunk 1/1: 100,000 samples
  Generating data...


    Epoch 1/10: Loss=0.2974, Train Acc=0.8948


Training:  40%|███▉      | 155/391 [00:43<01:00,  3.90it/s, loss=0.2995, acc=0.8963]

In [23]:
# ============================================================================
# QUICK TEST - Run this first to verify everything works
# ============================================================================

print("🧪 Quick Test: Training a small GNN on d=3")
print("="*50)

# Create trainer
test_trainer = GNNTrainer(p=0.005, d=3, device=device)

# Generate small dataset
print("\nGenerating 10,000 training samples...")
syndromes, labels = test_trainer.sample_data(10000)
print(f"✓ Syndromes shape: {syndromes.shape}")
print(f"✓ Labels shape: {labels.shape}")
print(f"✓ Example syndrome: {syndromes[0, :5].tolist()} ...")

# Train for 3 epochs
print("\nTraining for 3 epochs...")
for epoch in range(3):
    loss, acc = test_trainer.train_epoch(syndromes, labels, batch_size=256)
    print(f"  Epoch {epoch+1}: Loss={loss:.4f}, Accuracy={acc:.4f}")

# Evaluate
print("\nEvaluating on fresh test data...")
test_acc = test_trainer.evaluate(5000)
print(f"✓ Test Accuracy: {test_acc:.4f}")

print("\n" + "="*50)
print("✅ Quick test passed! The GNN is working.")
print("="*50)

🧪 Quick Test: Training a small GNN on d=3

Initializing GNN Trainer for d=3, p=0.005

[1/3] Building quantum circuit...

[2/3] Building graph skeleton...
✓ Graph skeleton built: 24 detectors, 48 connections
  Average neighbors per detector: 2.0

[3/3] Building neural network...
✓ Model built: 59,137 parameters

Ready to train!


Generating 10,000 training samples...
✓ Syndromes shape: torch.Size([10000, 24])
✓ Labels shape: torch.Size([10000])
✓ Example syndrome: [-1, -1, -1, -1, -1] ...

Training for 3 epochs...


  Epoch 1: Loss=0.4129, Accuracy=0.8907


  Epoch 2: Loss=0.2895, Accuracy=0.8960


  Epoch 3: Loss=0.2861, Accuracy=0.8960

Evaluating on fresh test data...
✓ Test Accuracy: 0.9024

✅ Quick test passed! The GNN is working.
